# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


**Lectura de la imagen.** Se toma la fila como primer índice y la columna como segundo. El inicio queda en (0,0) y las estanterías en (0,3), (1,1), (2,4) y (4,2). La entrega de valor 10 está en (0,5), la estación de carga de valor 2 en (2,2) y el peligro mortal en (3,5), los tres terminales. Los peligros no terminales de valor 3 negativo están en (1,4) y (4,1), y el piso resbaloso en (1,2), (2,1) y (3,3). Se agregan parámetros opcionales al constructor para facilitar los experimentos de la parte 5.

In [1]:
import numpy as np


class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slip_main=0.60):
        self.height = 5
        self.width = 6
        # Mapa tomado de la imagen del enunciado
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}
        self.terminal_states = {(0, 5): 10.0, (2, 2): 2.0, (3, 5): -10.0}
        self.danger_states = {(1, 4): -3.0, (4, 1): -3.0}
        self.living_reward = living_reward
        self.gamma = gamma
        self.slip_main = slip_main
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def move(self, state, action):
        # Si choca con borde o estantería permanece en el mismo estado
        nxt = (state[0] + action[0], state[1] + action[1])
        return nxt if self.is_valid_state(nxt) else state

    def get_transition_probs(self, state, action):
        # Los estados terminales son absorbentes
        if self.is_terminal(state):
            return [(state, 1.0)]
        if state in self.slippery_states:
            p_main = self.slip_main
        else:
            p_main = 0.90
        p_side = (1.0 - p_main) / 2
        # Desviaciones perpendiculares a la acción elegida
        left = (-action[1], action[0])
        right = (action[1], -action[0])
        probs = {}
        for a, p in ((action, p_main), (left, p_side), (right, p_side)):
            s2 = self.move(state, a)
            probs[s2] = probs.get(s2, 0.0) + p
        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("Todas las distribuciones de transición suman 1.")


Número de estados: 26
Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[s2] for s2, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
                V_new[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V, it


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for _ in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for it in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        changes = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changes)
        policy = new_policy
        if changes == 0:
            break
    V = policy_evaluation(grid, policy, threshold * 1e-3)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\nAmbos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [17, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas

**1.** Desde el inicio el robot se dirige a la estación de carga. La trayectoria greedy es derecha, derecha, abajo, abajo y termina en (2,2) con recompensa 2.

**2.** Todo camino hacia la entrega obliga a pasar por el peligro de (1,4), porque la estantería de (0,3) bloquea la fila superior y la ruta inferior termina en el peligro mortal. Sumando ese castigo, el costo por paso de varios movimientos y el descuento de 0.9, el valor esperado de la entrega resulta menor que el de la carga, que está a cuatro pasos.

**3.** Al comparar con un modelo sin piso resbaloso cambian las decisiones en (1,2) y (3,1). En (1,2) el robot elegiría ir hacia la derecha rumbo a la entrega si el piso fuera normal, pero con desviaciones del 40 % prefiere bajar a la carga.

**4.** El costo por paso castiga los recorridos largos. Con un valor de 1 por paso, los caminos largos hacia la entrega pierden atractivo frente a metas cercanas.

**5.** La probabilidad de moverse en la dirección elegida depende del tipo de piso del estado actual. Por eso T(s,a,s') se calcula consultando si s es resbaloso en lugar de usar una única tabla global.

In [7]:
def greedy_path(grid, policy, max_steps=30):
    s = grid.start
    path = [s]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        s = grid.move(s, policy[s])
        path.append(s)
    return path


def run_experiment(name, grid):
    V, n = value_iteration(grid)
    pi = extract_policy(grid, V)
    print(f"=== {name} ===")
    print("Iteraciones:", n)
    print_values(grid, V)
    print()
    print_policy(grid, pi)
    print("Trayectoria desde START:", greedy_path(grid, pi))
    print()
    return pi


pi_base = run_experiment("Base", WarehouseMDP())
pi_a = run_experiment("Experimento A, living_reward = -0.1", WarehouseMDP(living_reward=-0.1))
pi_b = run_experiment("Experimento B, piso resbaloso 0.40", WarehouseMDP(slip_main=0.40))
pi_c = run_experiment("Experimento C, gamma = 0.99", WarehouseMDP(gamma=0.99))

=== Base ===
Iteraciones: 20
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Trayectoria desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]

=== Experimento A, living_reward = -0.1 ===
Iteraciones: 26
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 | 

In [8]:
# Estados donde cambia la decisión si el piso resbaloso se comporta como normal
grid_normal = WarehouseMDP(slip_main=0.90)
V_normal, _ = value_iteration(grid_normal)
pi_normal = extract_policy(grid_normal, V_normal)
print([s for s in pi_base if pi_base[s] != pi_normal[s]])

[(1, 2), (3, 1)]


**Predicciones y resultados.** En el experimento A se predijo que al bajar el costo por paso el robot buscaría la entrega, y así ocurre. La trayectoria pasa por (1,2), (1,3), (1,4) y (1,5) hasta llegar a (0,5). En el experimento B se predijo que el robot seguiría prefiriendo la carga y se confirma, la política desde el inicio no cambia y solo bajan los valores cerca del piso resbaloso. En el experimento C, con gamma igual a 0.99, la recompensa lejana pierde menos valor por el descuento y el robot cambia su destino a la entrega.

In [9]:
for lr in np.arange(-1.0, -0.69, 0.01):
    grid_lr = WarehouseMDP(living_reward=lr)
    V_lr, _ = value_iteration(grid_lr)
    destino = greedy_path(grid_lr, extract_policy(grid_lr, V_lr))[-1]
    print(f"{lr:+.2f} -> {destino}")

-1.00 -> (2, 2)
-0.99 -> (2, 2)
-0.98 -> (2, 2)


-0.97 -> (2, 2)
-0.96 -> (2, 2)
-0.95 -> (2, 2)
-0.94 -> (2, 2)
-0.93 -> (2, 2)
-0.92 -> (2, 2)


-0.91 -> (2, 2)
-0.90 -> (2, 2)
-0.89 -> (2, 2)
-0.88 -> (2, 2)
-0.87 -> (2, 2)
-0.86 -> (2, 2)
-0.85 -> (2, 2)
-0.84 -> (2, 2)


-0.83 -> (2, 2)
-0.82 -> (2, 2)
-0.81 -> (2, 2)
-0.80 -> (2, 2)
-0.79 -> (0, 5)
-0.78 -> (0, 5)
-0.77 -> (0, 5)


-0.76 -> (0, 5)
-0.75 -> (0, 5)
-0.74 -> (0, 5)
-0.73 -> (0, 5)
-0.72 -> (0, 5)
-0.71 -> (0, 5)
-0.70 -> (0, 5)


-0.69 -> (0, 5)


**Bonus.** El cambio ocurre cerca de un living_reward de 0.80 negativo. Con valores menores o iguales el robot va a la carga y con valores mayores intenta llegar a la entrega.